In [1]:
import pandas as pd

In [2]:

# Path to parquet file
PARQUET_FILE_PATH = "./data/arxiv_papers/train.parquet"

In [3]:
# Load the parquet file
df = pd.read_parquet(PARQUET_FILE_PATH)
print(f"Loaded {len(df):,} total papers")

Loaded 2,549,619 total papers


In [4]:
# Count total number of rows in the DataFrame
print("Total row count:")
count_total = len(df)
print(count_total)

# Select all rows (entire DataFrame)
print("\nAll rows (first 5 shown):")
all_rows = df
print(all_rows.head())

# Count occurrences of each primary_subject value (excluding null values), sorted by count descending
print("\nCount of records by primary_subject (excluding nulls, sorted by count descending):")
result3 = (df[df['primary_subject'].notna()]
           .groupby('primary_subject')
           .size()
           .reset_index(name='total')
           .sort_values('total', ascending=False))
print(result3)

# Create character length buckets for abstract column and count distribution
print("\nDistribution of abstract character lengths by bucket:")
abstracts_not_null = df[df['abstract'].notna()].copy()
abstract_lengths = abstracts_not_null['abstract'].str.len()

# Define bins and labels for character length ranges
bins = [0, 1, 100, 200, 400, 800, 1600, 2000, 4096, float('inf')]
labels = ['0', '1-99', '100-199', '200-399', '400-799', '800-1599', '1600-1999', '2000-4095', '4096+']
bucket_order_map = {'0': 1, '1-99': 2, '100-199': 3, '200-399': 4, '400-799': 5, 
                   '800-1599': 6, '1600-1999': 7, '2000-4095': 8, '4096+': 9}

# Assign each abstract length to a bucket and order number
abstracts_not_null['bucket'] = pd.cut(abstract_lengths, bins=bins, labels=labels, right=False)
abstracts_not_null['bucket_order'] = abstracts_not_null['bucket'].map(bucket_order_map)

# Count total per bucket and calculate percentage of all abstracts
agg = abstracts_not_null.groupby(['bucket', 'bucket_order'], observed=False).size().reset_index(name='total')
agg['pct'] = round(100.0 * agg['total'] / agg['total'].sum(), 4)

# Final result sorted by bucket order
result4 = agg[['bucket', 'total', 'pct', 'bucket_order']].sort_values('bucket_order').reset_index(drop=True)
print(result4[['bucket', 'total', 'pct']])

Total row count:
2549619

All rows (first 5 shown):
    arxiv_id                                              title  \
0  0902.3253  The gravitational wave background from star-ma...   
1  0902.0428  Dynamics of planets in retrograde mean motion ...   
2  0901.3401  Diurnal Thermal Tides in a Non-synchronized Ho...   
3  0901.1570  Intermittent turbulence, noisy fluctuations an...   
4  0901.2048        Falling Transiting Extrasolar Giant Planets   

                                             authors  \
0       [Silvia Toonen, Clovis Hopman, Marc Freitag]   
1              [Julie Gayon, Eric Bois, Hans Scholl]   
2                    [Pin-Gao Gu, Gordon I. Ogilvie]   
3  [Z. Vörös, T. L. Zhang, M. P. Leubner, M. Volw...   
4        [B. Levrard, C. Winisdoerffer, G. Chabrier]   

                                     submission_date  \
0  18 Feb 2009 (<a href="https://arxiv.org/abs/09...   
1                                         3 Feb 2009   
2                                       